In [1]:
import pyrosetta
import numpy as np

from benchmark import bpti_ryfyn_benchmark
from misc import init_generator_params
from rotamer_extraction import extract_top_n_rotamers
from h_ising_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta
from custom_qaoa import qaoa_func_generator, run_qaoa
from h_mixer import custom_xy_mixer_layer

from constants import *
from validation import validate_conformations

initialize_rosetta(pyrosetta, extra_flags="-mute all")

Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
# Pyrosetta Relevant Code
benchmark_pose = bpti_ryfyn_benchmark()
rotamer_lib, ig, rot_sets, scorefxn = extract_top_n_rotamers(benchmark_pose)

# Generating QUBO (Quadratic Unconstrained Binary Optimisation) Model, and then reduce it
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
generator_params = init_generator_params(h_flex_linear)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

# Generate the actual observable and running functions we will use in the QAOA Algorithm
H_ising = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset)
cost_function, sample_function = qaoa_func_generator(H_ising, custom_xy_mixer_layer, generator_params)

# Run the Quantum Approximate Optimisation Algorithm and sample the final parameters
final_params = run_qaoa(cost_function)
probabilities = sample_function(final_params)

# Extract the top 100 most probably conformations and check that exactly 1 rotamer for each residue is selected
top_indices = list(np.argsort(probabilities)[-TOP_CONFORMATION_COUNTS:][::-1])
valid_conformations = validate_conformations(top_indices, probabilities, generator_params)

Fragment Sequence: RYFYN
Total Residues: 5
Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
{1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
initializing generator_params
{0: 1.0419909954071045, 1: 1.2052754163742065, 2: 1.2125394344329834, 3: 1.6054599285125732}
{0: 1.0419909954071045, 1: 1.2052754163742065, 2: 1.2125394344329834, 3: 1.6054599285125732}

---------------------------------------

{0: 1.4519609212875366, 1: 1.4519609212875366, 2: 2.4010090827941895, 3: 2.4010090827941895}
{0: 1.4519609212875366, 1: 1.4519609212875366, 2: 2.4010090827941895, 3: 2.4010090827941895}

---------------------------------------

{0: 1.4840223789215088, 1: 1.6721895933151245, 2: 2.909672975540161}
{0: 1.4840223789215088, 1: 1.6721895933151245, 2: 2.909672975540161}

---------------------------------------

{0: 1.84874999

In [10]:
import importlib, energy_calculation, validation
importlib.reload(validation)
importlib.reload(energy_calculation)

from energy_calculation import calculate_and_compare_energies
calculate_and_compare_energies(valid_conformations, h_flex_linear, J_flex_quadratic, global_offset, benchmark_pose, scorefxn, rotamer_lib, generator_params)

==================== ENERGY OPERATIONS ====================
Calculating Quantum Energies for all 100 conformations
Calculating Pyrosetta for all 100 conformations 
Comparing both energy types for all 100 conformations 
==================== ENERGY OPERATIONS COMPLETE ====================
